# QM Regression Workflow with QM7 Example

This notebook demonstrates a complete quantum-mechanics regression workflow using **kgcnn_torch** (PyTorch).

Steps:
1. Load and preprocess the QM7 dataset using `QMDataset`
2. Build molecular graphs with 3D coordinates and range-based edges
3. Build a SchNet model for energy prediction
4. Use `ExtensiveMolecularLabelScaler` for per-atom offset removal
5. Train with cross-validation
6. Evaluate with MAE/RMSE in original scale

In [ ]:
import os
import torch
import numpy as np
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

qm7_dir = "qm7"
qm7_csv = os.path.join(qm7_dir, "qm7.csv")
qm7_xyz = os.path.join(qm7_dir, "qm7.xyz")

if not os.path.exists(qm7_csv) or not os.path.exists(qm7_xyz):
    raise FileNotFoundError(
        "QM7 files not found at 'qm7/qm7.csv' and 'qm7/qm7.xyz'.\n"
        "Download them first with:\n"
        "from kgcnn.data.datasets.QM7Dataset import QM7Dataset\n"
        "QM7Dataset()\n"
        "Then copy the files to the local ./qm7/ folder."
    )

print(f"Found QM7 dataset at {qm7_dir}/")


## 1. Load the QM7 Dataset

QM7 contains 7165 molecules with atomization energies from DFT (PBE0/tier2 basis set).
The dataset provides XYZ coordinates and a CSV with labels.

If you have not yet downloaded QM7, use the Keras package to download:
```python
from kgcnn.data.datasets.QM7Dataset import QM7Dataset
QM7Dataset()
```
Then copy from `~/.kgcnn/datasets/QM7Dataset/` to a local `qm7/` folder.

In [ ]:
from kgcnn_torch.data.qm import QMDataset

data = QMDataset(
    data_directory="qm7/",
    dataset_name="qm7",
    file_name="qm7.csv",
    file_name_xyz="qm7.xyz"
)

## 2. Prepare Data

SchNet is a distance-based model that does not require bond/edge information from SDF files.
We read atomic coordinates and numbers directly from the XYZ file, and labels from the CSV.

In [ ]:
import pandas as pd

# Read atomic coordinates, symbols, and numbers directly from XYZ file.
# This bypasses SDF conversion which is unnecessary for distance-based models like SchNet.
data.read_in_memory_xyz()

# Load labels from CSV
df = pd.read_csv("qm7/qm7.csv")
labels_list = [np.atleast_1d(np.array(df.loc[i, "u0_atom"], dtype="float")) for i in range(len(df))]
data.assign_property("graph_labels", labels_list)

print("Number of molecules:", len(data))
print("First graph keys:", data[0].keys())
print("node_coordinates shape:", data[0]["node_coordinates"].shape)
print("node_number:", data[0]["node_number"])

## 3. Compute Range-Based Edges (Distance Cutoff)

SchNet uses distance-based neighbor lists instead of bond-based edges.
We compute range indices and distances for all atom pairs within a cutoff distance.

In [ ]:
from kgcnn_torch.graph.preprocessor import SetRange

# Compute range-based edges with 5 Angstrom cutoff.
# Use apply_preprocessor to properly merge results back into each graph dict.
set_range = SetRange(max_distance=5.0, overwrite=True)
for g in data:
    g.apply_preprocessor(set_range)

print("First graph updated keys:", data[0].keys())
print("Range indices shape:", data[0]["range_indices"].shape)
print("Range attributes (distances) shape:", data[0]["range_attributes"].shape)

## 4. Extract Labels and Prepare Cross-Validation

In [ ]:
labels = np.array(data.obtain_property("graph_labels"))
if len(labels.shape) <= 1:
    labels = np.expand_dims(labels, axis=-1)
print("Labels shape:", labels.shape)
print("Label range: [{:.2f}, {:.2f}]".format(labels.min(), labels.max()))

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, random_state=42, shuffle=True)
train_test_indices = [
    [train_index, test_index]
    for train_index, test_index in kf.split(X=np.zeros((len(data), 1)), y=labels)
]
print("Number of folds:", len(train_test_indices))

## 5. Convert to PyG Data and Configure SchNet

SchNet expects:
- `data.z`: Atomic numbers (used for node embedding)
- `data.pos`: Atom positions (used to compute distances internally)
- `data.edge_index`: Neighbor list indices
- `data.batch`: Batch assignment

We convert using `to_pyg_list()` with `edge_key="range_indices"` to use
the distance-based neighbor list instead of bond edges.

In [ ]:
pyg_list = data.to_pyg_list(edge_key="range_indices")

print("First PyG data object:")
print(pyg_list[0])
print("Atomic numbers (z):", pyg_list[0].z)
print("Positions shape:", pyg_list[0].pos.shape)
print("Edge index shape:", pyg_list[0].edge_index.shape)

In [ ]:
from kgcnn_torch.models.schnet import SchNetModel

model_kwargs = dict(
    node_dim=64,
    depth=4,
    units=128,
    gauss_bins=20,
    gauss_distance=4.0,
    gauss_sigma=0.4,
    gauss_offset=0.0,
    interaction_activation="shifted_softplus",
    interaction_pooling="sum",
    node_pooling="sum",
    last_mlp_units=[128, 64],
    last_mlp_activation="shifted_softplus",
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=True,
    num_embeddings=95,
    make_distance=True,
    expand_distance=True,
    use_output_mlp=False,
)

print("SchNet configured for QM regression")

## 6. Training Loop with Extensive Property Scaling

For QM properties like atomization energy, we use `ExtensiveMolecularLabelScaler`
which removes per-atom energy offsets via ridge regression on atom counts,
then scales the residuals. This is important for extensive properties that
scale with system size.

In [ ]:
import time
from datetime import timedelta
from torch_geometric.loader import DataLoader
from kgcnn_torch.training.trainer import fit
from kgcnn_torch.data.transform import ExtensiveMolecularLabelScaler

history_list = []
test_indices_list = []
model = None
scaler = None

for fold_idx, (train_index, test_index) in enumerate(train_test_indices):
    print(f"\n=== Fold {fold_idx} ===")

    # Create fresh model for each fold
    model = SchNetModel(**model_kwargs)
    model = model.to(device)

    # Get atomic numbers for scaler
    atoms_train = [data[i]["node_number"] for i in train_index]
    atoms_test = [data[i]["node_number"] for i in test_index]

    y_train = labels[train_index].copy()
    y_test = labels[test_index].copy()

    # Fit extensive scaler: removes per-atom energy offset
    scaler = ExtensiveMolecularLabelScaler(standardize_scale=True)
    scaler.fit(y_train, atomic_number=atoms_train)
    y_train_scaled = scaler.transform(y_train, atomic_number=atoms_train)
    y_test_scaled = scaler.transform(y_test, atomic_number=atoms_test)

    # Prepare PyG data with scaled labels
    train_data = [pyg_list[i].clone() for i in train_index]
    test_data = [pyg_list[i].clone() for i in test_index]
    for i in range(len(train_data)):
        train_data[i].y = torch.tensor(y_train_scaled[i], dtype=torch.float)
    for i in range(len(test_data)):
        test_data[i].y = torch.tensor(y_test_scaled[i], dtype=torch.float)

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    # Optimizer and scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0, end_factor=0.02, total_iters=800
    )
    loss_fn = torch.nn.L1Loss()  # MAE

    # Metrics in original scale
    def mae_metric(pred, target):
        return torch.mean(torch.abs(pred - target))

    def rmse_metric(pred, target):
        return torch.sqrt(torch.mean((pred - target) ** 2))

    metrics = {"mae": mae_metric, "rmse": rmse_metric}

    # Train (reduce epochs for the notebook demonstration)
    start = time.process_time()
    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=test_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        scheduler=scheduler,
        epochs=300,
        device=device,
        metrics=metrics,
        verbose=1,
        scaler=scaler,
    )
    stop = time.process_time()
    print(f"Training time: {timedelta(seconds=stop - start)}")

    history_list.append(history)
    test_indices_list.append([train_index, test_index])

## 7. Plot Training Curves

In [ ]:
from kgcnn_torch.utils.plots import plot_train_test_loss, plot_predict_true

plot_train_test_loss(
    history_list,
    loss_name="train_loss",
    val_loss_name="val_loss",
    model_name="SchNet",
    data_unit="kcal/mol",
    dataset_name="QM7",
    filepath=None,
    file_name="loss.png",
);

## 8. Prediction vs True (Last Fold)

We evaluate predictions in the original energy scale using the scaler's inverse transform.

In [ ]:
model.eval()
last_train_idx, last_test_idx = test_indices_list[-1]
test_data_last = [pyg_list[i] for i in last_test_idx]
test_loader_last = DataLoader(test_data_last, batch_size=32, shuffle=False)

all_preds = []
with torch.no_grad():
    for batch in test_loader_last:
        batch = batch.to(device)
        pred = model(batch)
        all_preds.append(pred.cpu().numpy())

predicted_y_scaled = np.concatenate(all_preds, axis=0)

# Inverse transform predictions back to original scale
atoms_test_last = [data[i]["node_number"] for i in last_test_idx]
predicted_y = scaler.inverse_transform(predicted_y_scaled, atomic_number=atoms_test_last)
true_y = labels[last_test_idx]

# Compute metrics
mae = np.mean(np.abs(predicted_y - true_y))
rmse = np.sqrt(np.mean((predicted_y - true_y) ** 2))
print(f"Test MAE: {mae:.4f} kcal/mol")
print(f"Test RMSE: {rmse:.4f} kcal/mol")

plot_predict_true(
    predicted_y, true_y,
    data_unit="kcal/mol",
    model_name="SchNet",
    dataset_name="QM7",
    filepath=None,
    file_name="predict.png",
    show_fig=True,
);

## Notes on Geometric Features

SchNet internally computes:
- **Interatomic distances** from `data.pos` and `data.edge_index`
- **Gaussian basis expansion** of distances (controlled by `gauss_bins`, `gauss_distance`, `gauss_sigma`)

For models like DimeNet++ that also need angular features, you would additionally
compute angle indices using `SetAngle` preprocessor:
```python
from kgcnn_torch.graph.preprocessor import SetAngle
data.map_list(SetAngle())
```